<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

In [4]:
import polars as pl
import plotly.express as px
import os, sys

module_path = os.path.abspath(os.path.join('../../../..')) # or the path to your source code
sys.path.insert(0, module_path)
import util

config = util.summary_settings
input_config = util.input_settings

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
trip = util.get_trip_data()
parcel_geog = util.get_parcel_geog()

df_trip = trip.join(parcel_geog, left_on='dpcl', right_on='ParcelID', how='left')

In [6]:
mode_cat = {1: "1: walk",
            2: "2: bike",
            3: "3: sov",
            4: "4: hov 2",
            5: "5: hov 3+",
            6: "6: transit",
            8: "8: school bus",
            9: "9:tnc"}
pdpurp_cat = {1: "1: Work",
              2: "2: School",
              3: "3: Escort",
              4: "4: other home-based",
              5: "4: other home-based",
              6: "4: other home-based",
              7: "4: other home-based",
              8: "4: other home-based",
              9: "4: other home-based",
              10: "4: other home-based"}

df_trip = df_trip.with_columns(
    mode_label=pl.col("mode").replace_strict(mode_cat, default=None),
    dpurp_label=pl.col("dpurp").replace_strict(pdpurp_cat, default=None))

In [7]:
def plot_mode_choice(df: pl.DataFrame, grp: str, title_name: str):
    df_plot = (
        df.group_by(['source', grp, 'mode_label'])
        .agg(
            trexpfac_sum=pl.col('trexpfac').sum(),
            sample_count=pl.col('trexpfac').count()
        )
        .with_columns(
            (pl.col('trexpfac_sum') / pl.col('trexpfac_sum').sum().over(['source', grp])).alias('percentage')
        )
    )

    fig = px.bar(
        df_plot.sort(by=[grp, 'source']),
        x="mode_label",
        y="percentage",
        color="source",
        facet_col=grp,
        facet_col_wrap=2,
        barmode="group",
        hover_data=['sample_count'],
        title=title_name
    )
    fig.update_layout(
        height=500,
        width=900,
        font=dict(size=11),
        xaxis=dict(dtick=1, categoryorder='category ascending')
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.for_each_yaxis(lambda a: a.update(tickformat=".1%"))
    fig.show()

In [8]:
df_trip = df_trip.filter(pl.col('dpurp_label') != 'None')
plot_mode_choice(df_trip,'dpurp_label', "trip mode choice by purpose")

In [9]:
df_trip = df_trip.filter(pl.col('CountyName').is_in(["King", "Kitsap", "Pierce", "Snohomish"]))
plot_mode_choice(df_trip,'CountyName', "trip mode choice by destination county")